# Model Training & Evaluation — energy_crisis TOPIC ONLY (standalone test)



In [1]:
!pip install -q transformers torch gensim scikit-learn statsmodels pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 27.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

# ---- CONFIG ----
DATA_DIR = "/content/drive/MyDrive/brexit_covid_datasets"
ECON_INDICATOR_PATH = f"{DATA_DIR}/economic_indicator.csv"

OUTLETS = ["bbc", "dailymail", "guardian"]
TOPICS = ["energy_crisis"]

AGG_FREQ = "MS"
LSTM_LOOKBACK = 3
TEST_FRACTION = 0.2
RANDOM_STATE = 42

# energy_crisis only has data from Aug 2021 onward, so only the 2023 H2 recession
# is even testable here (see markdown note above for why 2020 is left out).
RECESSION_WINDOWS = [
    {"name": "2023 H2 recession", "start": pd.Timestamp("2023-07-01"), "end": pd.Timestamp("2023-12-31")},
]

LEADUP_MONTHS = 6
CALM_COMPARISON_MONTHS = 6

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

Mounted at /content/drive


## Load data: energy_crisis articles per outlet, and the economic indicator

In [4]:
def load_outlet_topic(outlet, topic):
    df = pd.read_csv(f"{DATA_DIR}/{outlet}_energy_crisis_TEST.csv")
    df = df[df["topic"] == topic].copy()
    df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True).dt.tz_localize(None)
    df = df.dropna(subset=["publish_date", "text"]).sort_values("publish_date").reset_index(drop=True)
    return df

datasets = {}
for outlet in OUTLETS:
    for topic in TOPICS:
        key = (outlet, topic)
        try:
            df = load_outlet_topic(outlet, topic)
            datasets[key] = df
            print(f"{outlet}/{topic}: {len(df)} articles, {df['publish_date'].min()} to {df['publish_date'].max()}")
        except FileNotFoundError:
            print(f"{outlet}/{topic}: FILE NOT FOUND")

bbc/energy_crisis: 800 articles, 2021-08-06 00:00:00 to 2026-08-12 00:00:00
dailymail/energy_crisis: 799 articles, 2021-08-07 00:00:00 to 2026-03-16 00:00:00
guardian/energy_crisis: 800 articles, 2021-08-03 05:31:14 to 2026-08-13 14:54:03


In [5]:
econ = pd.read_csv(ECON_INDICATOR_PATH)
econ["date"] = pd.to_datetime(econ["date"])
econ = econ.set_index("date").resample(AGG_FREQ).mean().rename(columns={"value": "econ_value"})
print(econ.head())
print(f"Economic indicator range: {econ.index.min()} to {econ.index.max()}, {len(econ)} months")

            econ_value
date                  
1997-01-01        61.6
1997-02-01        61.9
1997-03-01        61.8
1997-04-01        62.6
1997-05-01        62.1
Economic indicator range: 1997-01-01 00:00:00 to 2026-05-01 00:00:00, 353 months


## Model 1 — FinBERT sentiment extraction

In [6]:
from transformers import pipeline

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    truncation=True,
    max_length=512,
    model_kwargs={"use_safetensors": False},
)

def finbert_signed_scores(texts, batch_size=16):
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = [t[:2000] if isinstance(t, str) else "" for t in texts[i:i+batch_size]]
        results = finbert(batch)
        for r in results:
            label, score = r["label"].lower(), r["score"]
            scores.append(score if label == "positive" else (-score if label == "negative" else 0.0))
    return scores

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Model 2 — LDA topic modeling

In [7]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

def fit_lda(texts, num_topics=5, passes=10):
    tokenized = [[w for w in simple_preprocess(str(t)) if w not in STOPWORDS and len(w) > 2] for t in texts]
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=2, no_above=0.5)
    corpus = [dictionary.doc2bow(t) for t in tokenized]
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=passes, random_state=RANDOM_STATE)
    return lda, dictionary, corpus

def lda_topic_scores(lda, corpus):
    scores = []
    for bow in corpus:
        dist = dict(lda.get_document_topics(bow, minimum_probability=0))
        scores.append(max(dist.values()) if dist else 0.0)
    return scores

In [8]:
EWMA_SPANS = {"1y": 365, "3m": 90}  # calendar days

def weighted_recency_features(df, score_col, prefix, agg_freq=AGG_FREQ):
    daily = df.set_index("publish_date")[score_col].resample("D").mean()
    daily = daily.interpolate(limit_direction="both")
    daily_change = daily.diff()

    out = pd.DataFrame(index=daily.index)
    for label, span in EWMA_SPANS.items():
        out[f"{prefix}_ewm_{label}"] = daily.ewm(span=span, adjust=False).mean()
        out[f"{prefix}_ewm_change_{label}"] = daily_change.ewm(span=span, adjust=False).mean()

    return out.resample(agg_freq).last()

## Model 3 — FinBERT-LSTM (sequence model)

In [9]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def train_lstm(X_train, y_train, epochs=150, lr=0.01, hidden_size=16, patience=15, val_fraction=0.2):
    torch.manual_seed(RANDOM_STATE)
    n = len(y_train)
    n_val = max(1, int(n * val_fraction)) if n >= 10 else 0

    if n_val > 0:
        X_fit, y_fit = X_train[:-n_val], y_train[:-n_val]
        X_val, y_val = X_train[-n_val:], y_train[-n_val:]
    else:
        X_fit, y_fit = X_train, y_train
        X_val, y_val = None, None

    model = SentimentLSTM(input_size=1, hidden_size=hidden_size)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    Xt = torch.tensor(X_fit).unsqueeze(-1)
    yt = torch.tensor(y_fit)

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improve = 0
    stopped_at = epochs
    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            train_loss_history.append(loss_fn(model(Xt), yt).item())

        if X_val is not None:
            with torch.no_grad():
                Xv = torch.tensor(X_val).unsqueeze(-1)
                yv = torch.tensor(y_val)
                val_loss = loss_fn(model(Xv), yv).item()
            val_loss_history.append(val_loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_without_improve = 0
            else:
                epochs_without_improve += 1
                if epochs_without_improve >= patience:
                    stopped_at = epoch + 1
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
        model._stopped_at_epoch = stopped_at
        model._train_loss_history = train_loss_history
        model._val_loss_history = val_loss_history
        model._train_var = float(np.var(y_fit))
        model._val_var = float(np.var(y_val)) if X_val is not None else None
        return model

    model._stopped_at_epoch = stopped_at
    model._train_loss_history = train_loss_history
    model._val_loss_history = val_loss_history
    return model

def predict_lstm(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X).unsqueeze(-1)).numpy()

## Baselines — naive persistence and ARIMA (news-free)

In [10]:
from statsmodels.tsa.arima.model import ARIMA

def naive_persistence_forecast(y_train, n_test):
    return np.full(n_test, y_train[-1])

def select_arima_order(y_train, candidate_orders=((1, 0, 0), (1, 0, 1))):
    # shrunk from 5 candidates to 2 — this was the main slowdown source in the
    # full run and isn't the thing being tested here.
    best_order, best_aic = (1, 0, 0), float("inf")
    for order in candidate_orders:
        try:
            fit = ARIMA(y_train, order=order).fit()
            if fit.aic < best_aic:
                best_aic = fit.aic
                best_order = order
        except Exception:
            continue
    return best_order

def fit_arima_baseline(y_train, n_test, order=None):
    if order is None:
        order = select_arima_order(y_train)
    try:
        model = ARIMA(y_train, order=order).fit()
        return np.asarray(model.forecast(steps=n_test)), order
    except Exception as e:
        print(f"ARIMA failed ({e}) — falling back to naive baseline")
        return naive_persistence_forecast(y_train, n_test), order

## Main loop — all models + baselines, energy_crisis only, predicting GDP GROWTH

In [11]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

RIDGE_ALPHA_GRID = (0.1, 0.3, 1.0, 3.0, 10.0, 30.0)

def tune_ridge_alpha(X_train, y_train, alphas=RIDGE_ALPHA_GRID, min_months_for_cv=24):
    n = len(y_train)
    if n < min_months_for_cv:
        return 1.0, {1.0: None}

    n_splits = min(5, max(2, n // 12))
    tscv = TimeSeriesSplit(n_splits=n_splits)

    scores = {}
    for alpha in alphas:
        fold_rmses = []
        for tr_idx, val_idx in tscv.split(X_train):
            if len(tr_idx) < 3 or len(val_idx) < 1:
                continue
            m = Ridge(alpha=alpha).fit(X_train[tr_idx], y_train[tr_idx])
            pred = m.predict(X_train[val_idx])
            fold_rmses.append(mean_squared_error(y_train[val_idx], pred) ** 0.5)
        scores[alpha] = np.mean(fold_rmses) if fold_rmses else np.inf

    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    direction_match = np.sign(y_true) == np.sign(y_pred)
    return {
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan"),
        "directional_accuracy": direction_match.mean(),
    }

results = []

for (outlet, topic), df in datasets.items():
    print(f"\n=== {outlet} / {topic} ({len(df)} articles) ===")
    if len(df) < 20:
        print("Too few articles to model meaningfully — skipping.")
        continue

    df["finbert_score"] = finbert_signed_scores(df["text"].tolist())
    monthly_sent = df.set_index("publish_date")["finbert_score"].resample(AGG_FREQ).mean()

    lda, dictionary, corpus = fit_lda(df["text"].tolist())
    df["lda_score"] = lda_topic_scores(lda, corpus)
    monthly_lda = df.set_index("publish_date")["lda_score"].resample(AGG_FREQ).mean()

    weighted_feats = weighted_recency_features(df, "finbert_score", "finbert")
    lda_weighted_feats = weighted_recency_features(df, "lda_score", "lda")
    merged = pd.DataFrame({"finbert": monthly_sent, "lda": monthly_lda}) \
        .join(weighted_feats, how="left") \
        .join(lda_weighted_feats, how="left") \
        .join(econ, how="inner")
    merged = merged.ffill().dropna()
    merged["growth"] = merged["econ_value"].pct_change() * 100
    merged = merged.dropna()
    n_months = len(merged)
    print(f"Merged monthly series: {n_months} months")
    if n_months < 12:
        print("Fewer than 12 months of overlapping data — skipping.")
        continue

    split_idx = int(n_months * (1 - TEST_FRACTION))
    y = merged["growth"].values

    X1 = merged[["finbert"]].values
    alpha1, _ = tune_ridge_alpha(X1[:split_idx], y[:split_idx])
    m1 = Ridge(alpha=alpha1).fit(X1[:split_idx], y[:split_idx])
    pred1 = m1.predict(X1[split_idx:])
    results.append({"outlet": outlet, "topic": topic, "model": "FinBERT+Ridge", **evaluate(y[split_idx:], pred1), "n_months": n_months, "tuned_param": f"alpha={alpha1}"})

    X2 = merged[["lda"]].values
    alpha2, _ = tune_ridge_alpha(X2[:split_idx], y[:split_idx])
    m2 = Ridge(alpha=alpha2).fit(X2[:split_idx], y[:split_idx])
    pred2 = m2.predict(X2[split_idx:])
    results.append({"outlet": outlet, "topic": topic, "model": "LDA+Ridge", **evaluate(y[split_idx:], pred2), "n_months": n_months, "tuned_param": f"alpha={alpha2}"})

    X_combined = merged[["finbert", "lda"]].values
    alpha_c, _ = tune_ridge_alpha(X_combined[:split_idx], y[:split_idx])
    m_combined = Ridge(alpha=alpha_c).fit(X_combined[:split_idx], y[:split_idx])
    pred_combined = m_combined.predict(X_combined[split_idx:])
    results.append({"outlet": outlet, "topic": topic, "model": "FinBERT+LDA+Ridge", **evaluate(y[split_idx:], pred_combined), "n_months": n_months, "tuned_param": f"alpha={alpha_c}"})

    weighted_cols = [c for c in merged.columns if c.startswith("finbert_ewm")]
    X_weighted = merged[["finbert"] + weighted_cols].values
    alpha_w, _ = tune_ridge_alpha(X_weighted[:split_idx], y[:split_idx])
    m_weighted = Ridge(alpha=alpha_w).fit(X_weighted[:split_idx], y[:split_idx])
    pred_weighted = m_weighted.predict(X_weighted[split_idx:])
    results.append({"outlet": outlet, "topic": topic, "model": "FinBERT+WeightedMA+Ridge", **evaluate(y[split_idx:], pred_weighted), "n_months": n_months, "tuned_param": f"alpha={alpha_w}"})

    lda_weighted_cols = [c for c in merged.columns if c.startswith("lda_ewm")]
    X_lda_weighted = merged[["lda"] + lda_weighted_cols].values
    alpha_lw, _ = tune_ridge_alpha(X_lda_weighted[:split_idx], y[:split_idx])
    m_lda_weighted = Ridge(alpha=alpha_lw).fit(X_lda_weighted[:split_idx], y[:split_idx])
    pred_lda_weighted = m_lda_weighted.predict(X_lda_weighted[split_idx:])
    results.append({"outlet": outlet, "topic": topic, "model": "LDA+WeightedMA+Ridge", **evaluate(y[split_idx:], pred_lda_weighted), "n_months": n_months, "tuned_param": f"alpha={alpha_lw}"})

    series = merged["finbert"].values.astype(np.float32)
    growth_series = merged["growth"].values.astype(np.float32)
    if n_months - LSTM_LOOKBACK < 12:
        print("Not enough months for LSTM lookback — skipping Model 3.")
    else:
        Xs, ys = [], []
        for i in range(len(series) - LSTM_LOOKBACK):
            Xs.append(series[i:i + LSTM_LOOKBACK])
            ys.append(growth_series[i + LSTM_LOOKBACK])
        Xs, ys = np.array(Xs), np.array(ys)
        split_seq = int(len(Xs) * (1 - TEST_FRACTION))

        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(ys[:split_seq].reshape(-1, 1)).flatten()
        lstm_model = train_lstm(Xs[:split_seq], y_train_scaled)
        train_r2 = 1 - lstm_model._train_loss_history[-1] / lstm_model._train_var
        val_r2 = 1 - lstm_model._val_loss_history[-1] / lstm_model._val_var
        print(f"{outlet}/{topic} LSTM — train_R2={train_r2:.3f}, val_R2={val_r2:.3f}")
        pred3_scaled = predict_lstm(lstm_model, Xs[split_seq:])
        pred3 = y_scaler.inverse_transform(pred3_scaled.reshape(-1, 1)).flatten()
        results.append({"outlet": outlet, "topic": topic, "model": "FinBERT-LSTM", **evaluate(ys[split_seq:], pred3), "n_months": len(Xs), "tuned_param": f"stopped_epoch={lstm_model._stopped_at_epoch}"})

    naive_pred = naive_persistence_forecast(y[:split_idx], len(y) - split_idx)
    results.append({"outlet": outlet, "topic": topic, "model": "Naive (no-change)", **evaluate(y[split_idx:], naive_pred), "n_months": n_months, "tuned_param": ""})

    arima_pred, arima_order = fit_arima_baseline(y[:split_idx], len(y) - split_idx)
    results.append({"outlet": outlet, "topic": topic, "model": "ARIMA(auto)", **evaluate(y[split_idx:], arima_pred), "n_months": n_months, "tuned_param": f"order={arima_order}"})

results_df = pd.DataFrame(results)
print("\n\n=== FINAL RESULTS (energy_crisis, predicting GDP growth, with baselines) ===")
print(results_df)
results_df.to_csv(f"{DATA_DIR}/model_results_energy_crisis_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/model_results_energy_crisis_TEST.csv")


=== bbc / energy_crisis (800 articles) ===
Merged monthly series: 57 months
bbc/energy_crisis LSTM — train_R2=0.001, val_R2=-0.126

=== dailymail / energy_crisis (799 articles) ===
Merged monthly series: 55 months
dailymail/energy_crisis LSTM — train_R2=0.025, val_R2=-0.071

=== guardian / energy_crisis (800 articles) ===
Merged monthly series: 57 months
guardian/energy_crisis LSTM — train_R2=0.095, val_R2=-0.239


=== FINAL RESULTS (energy_crisis, predicting GDP growth, with baselines) ===
       outlet          topic                     model      rmse       mae  \
0         bbc  energy_crisis             FinBERT+Ridge  0.228542  0.192145   
1         bbc  energy_crisis                 LDA+Ridge  0.220954  0.182561   
2         bbc  energy_crisis         FinBERT+LDA+Ridge  0.234764  0.196920   
3         bbc  energy_crisis  FinBERT+WeightedMA+Ridge  0.226645  0.190101   
4         bbc  energy_crisis      LDA+WeightedMA+Ridge  0.243110  0.184638   
5         bbc  energy_crisis       

## Recession-specific test — 2023 H2 only



In [12]:
recession_results = []

for outlet in OUTLETS:
    key = (outlet, "energy_crisis")
    if key not in datasets:
        continue
    df = datasets[key]
    if "finbert_score" not in df.columns:
        print(f"{outlet}/energy_crisis: run the main loop cell first — finbert_score not found.")
        continue

    monthly_sent = df.set_index("publish_date")["finbert_score"].resample(AGG_FREQ).mean()
    merged = pd.DataFrame({"finbert": monthly_sent}).join(econ, how="inner")
    merged = merged.ffill().dropna()
    merged["growth"] = merged["econ_value"].pct_change() * 100
    merged = merged.dropna()

    for window in RECESSION_WINDOWS:
        rec_start, rec_end, label = window["start"], window["end"], window["name"]

        train = merged[merged.index < rec_start]
        leadup_start = rec_start - pd.DateOffset(months=LEADUP_MONTHS)
        test_leadup = merged[(merged.index >= leadup_start) & (merged.index < rec_start)]
        test_recession = merged[(merged.index >= rec_start) & (merged.index <= rec_end)]
        calm_end = rec_end + pd.DateOffset(months=CALM_COMPARISON_MONTHS)
        test_calm = merged[(merged.index > rec_end) & (merged.index <= calm_end)]

        print(f"{outlet}/energy_crisis [{label}]: {len(train)} training months, "
              f"{len(test_leadup)} lead-up months, {len(test_recession)} recession months, "
              f"{len(test_calm)} calm-comparison months")

        if len(train) < 24 or len(test_recession) < 2:
            print(f"  -> not enough data for {label}, skipping {outlet}/energy_crisis")
            continue

        alpha_r, _ = tune_ridge_alpha(train[["finbert"]].values, train["growth"].values)
        print(f"  tuned alpha for {outlet}/{label}: {alpha_r}")
        m = Ridge(alpha=alpha_r).fit(train[["finbert"]].values, train["growth"].values)

        for period_name, test_set in [
            ("lead-up (pre-recession)", test_leadup),
            ("recession", test_recession),
            ("calm (post-recession)", test_calm),
        ]:
            if len(test_set) < 2:
                continue
            pred = m.predict(test_set[["finbert"]].values)
            recession_results.append({
                "outlet": outlet,
                "recession": label,
                "period": period_name,
                **evaluate(test_set["growth"].values, pred),
                "n_months": len(test_set),
                "tuned_param": f"alpha={alpha_r}",
            })

recession_df = pd.DataFrame(recession_results)
print("\n\n=== LEAD-UP vs RECESSION vs CALM COMPARISON (energy_crisis, FinBERT+Ridge) ===")
print(recession_df)
recession_df.to_csv(f"{DATA_DIR}/recession_specific_results_energy_crisis_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/recession_specific_results_energy_crisis_TEST.csv")

bbc/energy_crisis [2023 H2 recession]: 22 training months, 6 lead-up months, 6 recession months, 6 calm-comparison months
  -> not enough data for 2023 H2 recession, skipping bbc/energy_crisis
dailymail/energy_crisis [2023 H2 recession]: 22 training months, 6 lead-up months, 6 recession months, 6 calm-comparison months
  -> not enough data for 2023 H2 recession, skipping dailymail/energy_crisis
guardian/energy_crisis [2023 H2 recession]: 22 training months, 6 lead-up months, 6 recession months, 6 calm-comparison months
  -> not enough data for 2023 H2 recession, skipping guardian/energy_crisis


=== LEAD-UP vs RECESSION vs CALM COMPARISON (energy_crisis, FinBERT+Ridge) ===
Empty DataFrame
Columns: []
Index: []

Saved to /content/drive/MyDrive/brexit_covid_datasets/recession_specific_results_energy_crisis_TEST.csv


## Clear, separated results tables

In [13]:
print("=== MAIN RESULTS, by outlet ===\n")
for (outlet, topic), sub in results_df.groupby(["outlet", "topic"]):
    print(f"--- {outlet} / {topic} ---")
    print(sub[["model", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
    print()

if len(recession_df):
    print("\n=== RECESSION-SPECIFIC RESULTS, by outlet (2023 H2 only) ===\n")
    for (outlet, rec), sub in recession_df.groupby(["outlet", "recession"]):
        print(f"--- {outlet} / {rec} ---")
        print(sub[["period", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
        print()
else:
    print("\nNo outlet had enough pre-recession training months (24+) for the recession-specific test — "
          "see the risk noted at the top of this notebook.")

=== MAIN RESULTS, by outlet ===

--- bbc / energy_crisis ---
                   model     rmse      mae        r2  directional_accuracy  n_months      tuned_param
           FinBERT+Ridge 0.228542 0.192145 -0.195241              0.500000        57        alpha=1.0
               LDA+Ridge 0.220954 0.182561 -0.117191              0.500000        57        alpha=0.3
       FinBERT+LDA+Ridge 0.234764 0.196920 -0.261211              0.500000        57        alpha=1.0
FinBERT+WeightedMA+Ridge 0.226645 0.190101 -0.175487              0.500000        57        alpha=1.0
    LDA+WeightedMA+Ridge 0.243110 0.184638 -0.352479              0.500000        57        alpha=0.1
            FinBERT-LSTM 0.199216 0.159500 -0.004112              0.454545        54 stopped_epoch=23
       Naive (no-change) 0.292056 0.220230 -0.951895              0.250000        57                 
             ARIMA(auto) 0.199784 0.163564  0.086635              0.500000        57  order=(1, 0, 0)

--- dailymail / ener